In [ ]:
# Benin Solar Dataset: Preprocessing

# Preprocessing and feature engineering for Benin solar dataset (10 Academy Solar Challenge Week 1).

# Objectives:
# - Handle missing values.
# - Normalize numerical features.
# - Engineer features (e.g., daily GHI average).
# - Save processed dataset.

# === Import Libraries ===
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Set random seed
np.random.seed(42)

# === Load Data ===
df = pd.read_csv('../data/benin.csv', parse_dates=['Timestamp'])
df.head()

# === Data Cleaning ===

# Convert 'Cleaning' to boolean
df['Cleaning'] = df['Cleaning'].astype(bool)

# Clip negative values in key columns
for col in ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'Precipitation']:
    df[col] = df[col].clip(lower=0)

# Handle missing values:
# Fill numerical columns with median
numerical_cols = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'Tamb', 'RH', 'WS', 'WSgust',
                  'WSstdev', 'WD', 'WDstdev', 'BP', 'Precipitation', 'TModA', 'TModB']
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill 'Comments' with 'None'
df['Comments'] = df['Comments'].fillna('None')

# Verify missing values handled
print("Missing Values:")
print(df.isnull().sum())

# === Feature Engineering ===

# Extract date features
df['Year'] = df['Timestamp'].dt.year
df['Month'] = df['Timestamp'].dt.month
df['Day'] = df['Timestamp'].dt.day
df['Hour'] = df['Timestamp'].dt.hour

# Compute Daily GHI Average
df['Date'] = df['Timestamp'].dt.date
daily_ghi = df.groupby('Date')['GHI'].mean().reset_index()
daily_ghi.columns = ['Date', 'Daily_GHI_Avg']
df = df.merge(daily_ghi, on='Date', how='left')
df = df.drop('Date', axis=1)

# === Feature Normalization ===

# Normalize numerical features
scaler = StandardScaler()
scaled_cols = numerical_cols + ['Daily_GHI_Avg']
df[scaled_cols] = scaler.fit_transform(df[scaled_cols])

# === Save Processed Data ===

# Ensure output directory exists
os.makedirs('notebooks', exist_ok=True)

# Save processed dataset
df.to_csv('notebooks/benin_processed.csv', index=False)

# Save summary of new features
new_features = ['Year', 'Month', 'Day', 'Hour', 'Daily_GHI_Avg']
feature_summary = df[new_features].describe()
feature_summary.to_csv('notebooks/benin_feature_summary.csv')

# Display confirmation
print("✅ Processing complete. Files saved to 'notebooks/'.")


Missing Values:
Timestamp        0
GHI              0
DNI              0
DHI              0
ModA             0
ModB             0
Tamb             0
RH               0
WS               0
WSgust           0
WSstdev          0
WD               0
WDstdev          0
BP               0
Cleaning         0
Precipitation    0
TModA            0
TModB            0
Comments         0
dtype: int64


OSError: Cannot save file into a non-existent directory: 'notebooks'